In [0]:
# Native without Spark. Separator, quotes, BOM, header
print(dbutils.fs.head("/Volumes/imaginex/nba/source_data/nba2k-full.csv", 600))

[Truncated to first 600 bytes]
full_name,rating,jersey,team,position,b_day,height,weight,salary,country,draft_year,draft_round,draft_peak,college,version
LeBron James,97,#23,Los Angeles Lakers,F,12/30/84,6-9 / 2.06,250 lbs. / 113.4 kg.,$37436858,USA,2003,1,1,,NBA2k20
Kawhi Leonard,97,#2,Los Angeles Clippers,F,06/29/91,6-7 / 2.01,225 lbs. / 102.1 kg.,$32742000,USA,2011,1,15,San Diego State,NBA2k20
Giannis Antetokounmpo,96,#34,Milwaukee Bucks,F-G,12/06/94,6-11 / 2.11,242 lbs. / 109.8 kg.,$25842697,Greece,2013,1,15,,NBA2k20
Kevin Durant,96,#7,Brooklyn Nets,F,09/29/88,6-10 / 2.08,230 lbs. / 104.3 kg.,$37199000,USA,2007,1,2,Texa


In [0]:
# All text as text, no hide problems
df = (spark.read
      .option("header", True)
      .option("inferSchema", False)
      .csv("/Volumes/imaginex/nba/source_data/nba2k-full.csv"))
print(df.count(), "raws,", len(df.columns), "columns-fields")
df.printSchema()

449 raws, 15 columns-fields
root
 |-- full_name: string (nullable = true)
 |-- rating: string (nullable = true)
 |-- jersey: string (nullable = true)
 |-- team: string (nullable = true)
 |-- position: string (nullable = true)
 |-- b_day: string (nullable = true)
 |-- height: string (nullable = true)
 |-- weight: string (nullable = true)
 |-- salary: string (nullable = true)
 |-- country: string (nullable = true)
 |-- draft_year: string (nullable = true)
 |-- draft_round: string (nullable = true)
 |-- draft_peak: string (nullable = true)
 |-- college: string (nullable = true)
 |-- version: string (nullable = true)



In [0]:
display(df.summary())

summary,full_name,rating,jersey,team,position,b_day,height,weight,salary,country,draft_year,draft_round,draft_peak,college,version
count,449,449,449,426,449,449,449,449,449,449,449,449,449,376,449
mean,null,77.20489977728285,null,null,null,null,null,null,null,null,2013.9510022271716,1.2774869109947644,20.845549738219894,null,null
stddev,null,6.748205651135889,null,null,null,null,null,null,null,null,3.992996673026217,0.4483460050543277,15.311283424020843,null,null
min,Aaron Gordon,67,#0,Atlanta Hawks,C,01/01/93,5-10 / 1.78,170 lbs. / 77.1 kg.,$100000,Angola,2001,1,1,Alabama,NBA2k20
25%,null,72.0,null,null,null,null,null,null,null,null,2011.0,1.0,7.0,null,null
50%,null,75.0,null,null,null,null,null,null,null,null,2015.0,1.0,18.0,null,null
75%,null,79.0,null,null,null,null,null,null,null,null,2017.0,2.0,32.0,null,null
max,Zion Williamson,97,#99,Washington Wizards,G-F,12/30/84,7-4 / 2.24,290 lbs. / 131.5 kg.,$9881598,United Kingdom,2019,Undrafted,Undrafted,Yale,NBA2k22


In [0]:
"""
Structural profile of a CSV in a Unity Catalog volume. Runs on serverless.
Parameters: CSV PATH and KEY (candidate key columns, comma-separated).
"""
import re
from pyspark.sql import DataFrame, functions as F

PATH = "/Volumes/imaginex/nba/source_data/nba2k-full.csv"
KEY = ["full_name", "b_day"]

# If running from your machine with Databricks Connect, uncomment these two lines:
# from databricks.connect import DatabricksSession
# spark = DatabricksSession.builder.profile("dev").serverless(True).getOrCreate()


def section(title: str) -> None:
    print(f"\n{'=' * 70}\n{title}\n{'=' * 70}")


def load_all_string(path: str) -> DataFrame:
    return spark.read.option("header", True).option("inferSchema", False).csv(path)


def pattern_expr(c: str):
    """Abstracts the format: digits -> 9, letters -> a. Same as the pandas version."""
    return F.regexp_replace(F.regexp_replace(F.col(c), r"\d", "9"), r"[A-Za-z]", "a")


df = load_all_string(PATH)
COLS = df.columns  # Cache once: avoids repeated Analyze RPCs in Spark Connect
df = df.select([F.when(F.trim(F.col(c)) == "", None).otherwise(F.col(c)).alias(c) for c in COLS])
n = df.count()

section("1. SHAPE")
print(f"rows: {n:,}   columns: {len(COLS)}")
print("columns:", COLS)

section("2. PER COLUMN: nulls, cardinality, patterns")
# Single pass with aggregations to avoid launching a job per column
aggs = []
for c in COLS:
    aggs += [
        F.sum(F.col(c).isNull().cast("int")).alias(f"{c}__nulls"),
        F.countDistinct(c).alias(f"{c}__distinct"),
        F.countDistinct(pattern_expr(c)).alias(f"{c}__patterns"),
    ]
_row = df.agg(*aggs).first()
assert _row is not None, "DataFrame is empty"
row = _row.asDict()

summary = spark.createDataFrame(
    [(c, row[f"{c}__nulls"], round(100 * row[f"{c}__nulls"] / n, 1), row[f"{c}__distinct"], row[f"{c}__patterns"])
     for c in COLS],
    ["column", "nulls", "pct_nulls", "unique", "patterns"],
)
summary.show(len(COLS), truncate=False)

section("3. PATTERNS PER COLUMN (only columns with more than one format)")
for c in COLS:
    if 1 < row[f"{c}__patterns"] <= 12:
        print(f"\n  {c}:")
        (df.filter(F.col(c).isNotNull())
           .groupBy(pattern_expr(c).alias("pattern"))
           .agg(F.count("*").alias("n"), F.first(c).alias("example"))
           .orderBy(F.desc("n"))
           .show(12, truncate=False))

section("4. CATEGORICAL (<= 15 unique values)")
for c in COLS:
    if 1 < row[f"{c}__distinct"] <= 15:
        print(f"\n  {c}:")
        df.groupBy(c).count().orderBy(F.desc("count")).show(15, truncate=False)

section("5. DUPLICATES")
print("identical full rows:", n - df.dropDuplicates().count())
dupes = (df.groupBy(*KEY).count().filter("count > 1"))
n_groups = dupes.count()
print(f"duplicate groups on key {KEY}: {n_groups}")
if n_groups:
    (df.join(dupes.select(*KEY), KEY).orderBy(*KEY).show(20, truncate=False))
print("columns that are key by themselves (unique == rows):",
      [c for c in COLS if row[f"{c}__distinct"] == n] or "none")

section("6. NUMERIC RANGES (after removing $ # and commas)")
for c in COLS:
    cleaned = F.expr(f"try_cast(regexp_replace({c}, '[$#,]', '') as double)")
    stats = df.agg(
        F.min(cleaned).alias("min"), F.expr(f"percentile_approx(try_cast(regexp_replace({c}, '[$#,]', '') as double), 0.5)").alias("p50"),
        F.max(cleaned).alias("max"),
        F.sum((cleaned.isNull() & F.col(c).isNotNull()).cast("int")).alias("non_numeric"),
        F.sum(cleaned.isNotNull().cast("int")).alias("numeric"),
    ).first()
    if stats["numeric"] > n * 0.5:
        print(f"  {c:<12} min={stats['min']:<14g} p50={stats['p50']!s:<14} max={stats['max']:<14g} "
              f"non-numeric={stats['non_numeric']}")

section("7. SAMPLE")
df.sample(fraction=0.02, seed=1).show(8, truncate=False)


1. SHAPE
rows: 449   columns: 15
columns: ['full_name', 'rating', 'jersey', 'team', 'position', 'b_day', 'height', 'weight', 'salary', 'country', 'draft_year', 'draft_round', 'draft_peak', 'college', 'version']

2. PER COLUMN: nulls, cardinality, patterns
+-----------+-----+---------+------+--------+
|column     |nulls|pct_nulls|unique|patterns|
+-----------+-----+---------+------+--------+
|full_name  |0    |0.0      |429   |84      |
|rating     |0    |0.0      |31    |1       |
|jersey     |0    |0.0      |52    |2       |
|team       |23   |5.1      |30    |24      |
|position   |0    |0.0      |7     |2       |
|b_day      |0    |0.0      |415   |1       |
|height     |0    |0.0      |20    |2       |
|weight     |0    |0.0      |84    |4       |
|salary     |0    |0.0      |325   |5       |
|country    |0    |0.0      |39    |15      |
|draft_year |0    |0.0      |18    |1       |
|draft_round|0    |0.0      |3     |2       |
|draft_peak |0    |0.0      |57    |3       |
|colleg